In [1]:
# Install the official Kaggle API client
!pip install kaggle

In [2]:
from google.colab import files

print("Please select your 'kaggle.json' file to upload:")
files.upload()
# A file selector box will appear. Click 'Choose Files' and select the kaggle.json file.

Please select your 'kaggle.json' file to upload:


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"gangulasomashekar","key":"324fab117239ff9bc1b5060e1a2215c8"}'}

In [3]:
import os

# 1. Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# 2. Move the uploaded kaggle.json file into the .kaggle directory
!mv kaggle.json ~/.kaggle/

# 3. Set permissions: The file must be read-only for security (owner only)
!chmod 600 ~/.kaggle/kaggle.json

print("\nKaggle API Key setup complete! You are now authenticated.")

# Verify the file is in place and permissions are correct (optional)
!ls -l ~/.kaggle/


Kaggle API Key setup complete! You are now authenticated.
total 4
-rw------- 1 root root 73 Nov 11 17:46 kaggle.json


In [4]:
import os

# 1. Define the desired folder name
DOWNLOAD_PATH = './clinical_data'

# 2. Create the folder if it doesn't exist
# The -p flag in !mkdir ensures no error is thrown if the directory already exists
!mkdir -p {DOWNLOAD_PATH}
print(f"Directory '{DOWNLOAD_PATH}' created.")

# 3. Download the dataset into the specified folder using the -p argument
# The -d flag specifies the dataset, and the -p flag specifies the path.
!kaggle datasets download -d azmayensabil/doctor-patient-conversation-large -p {DOWNLOAD_PATH}
print("Download complete.")

# 4. Unzip the downloaded file inside the target folder
# The zip file will be located at: ./clinical_data/doctor-patient-conversation-large.zip
ZIP_FILE_PATH = os.path.join(DOWNLOAD_PATH, 'doctor-patient-conversation-large.zip')

# -q for quiet (optional), -d for destination directory
!unzip -q {ZIP_FILE_PATH} -d {DOWNLOAD_PATH}
print(f"Unzip complete. Files are extracted to: {DOWNLOAD_PATH}")

# 5. (Optional) List the contents of the folder to confirm the download and unzip
print("\n--- Folder Contents ---")
!ls {DOWNLOAD_PATH}

Directory './clinical_data' created.
Dataset URL: https://www.kaggle.com/datasets/azmayensabil/doctor-patient-conversation-large
License(s): unknown
  0% 0.00/786k [00:00<?, ?B/s]
100% 786k/786k [00:00<00:00, 357MB/s]
Download complete.
Unzip complete. Files are extracted to: ./clinical_data

--- Folder Contents ---
CAR0001.txt			       RES0010.txt  RES0081.txt  RES0151.txt
CAR0002.txt			       RES0011.txt  RES0082.txt  RES0152.txt
CAR0003.txt			       RES0012.txt  RES0083.txt  RES0153.txt
CAR0004.txt			       RES0013.txt  RES0084.txt  RES0154.txt
CAR0005.txt			       RES0014.txt  RES0085.txt  RES0155.txt
DER0001.txt			       RES0015.txt  RES0086.txt  RES0156.txt
doctor-patient-conversation-large.zip  RES0016.txt  RES0087.txt  RES0158.txt
GAS0001.txt			       RES0017.txt  RES0088.txt  RES0159.txt
GAS0002.txt			       RES0018.txt  RES0089.txt  RES0160.txt
GAS0003.txt			       RES0019.txt  RES0090.txt  RES0161.txt
GAS0004.txt			       RES0020.txt  RES0091.txt  RES0162.txt
GAS0005.txt			 

In [5]:
import os
import sys
import json
import re
import argparse
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict, Counter
from datetime import datetime
from dataclasses import dataclass
from tqdm import tqdm
import pandas as pd

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# COMPREHENSIVE MEDICAL DICTIONARIES
# ============================================================================

class MedicalDictionary:
    """Comprehensive medical dictionaries for accurate entity extraction"""

    def __init__(self):
        self.symptoms = {
            # Pain and discomfort
            'pain', 'ache', 'soreness', 'discomfort', 'tenderness', 'sore',
            'headache', 'migraine', 'chest pain', 'abdominal pain', 'stomach pain',
            'back pain', 'neck pain', 'joint pain', 'muscle pain', 'body aches',

            # Respiratory
            'cough', 'coughing', 'dry cough', 'wet cough', 'productive cough',
            'shortness of breath', 'difficulty breathing', 'wheezing', 'wheeze',
            'chest tightness', 'runny nose', 'nasal congestion', 'stuffy nose',
            'sore throat', 'throat pain', 'hoarse voice', 'hoarseness', 'lost voice',
            'phlegm', 'sputum', 'mucus',

            # Systemic
            'fever', 'chills', 'sweating', 'night sweats', 'fatigue', 'tiredness',
            'weakness', 'malaise', 'lethargy', 'exhaustion',

            # GI symptoms
            'nausea', 'nauseous', 'vomiting', 'vomit', 'throwing up',
            'diarrhea', 'diarrhoea', 'constipation', 'abdominal pain', 'stomach pain',
            'bloating', 'gas', 'indigestion', 'heartburn', 'acid reflux',

            # Neurological
            'dizziness', 'vertigo', 'lightheadedness', 'fainting', 'syncope',
            'numbness', 'tingling', 'paresthesia',

            # Other common symptoms
            'rash', 'itching', 'pruritus', 'hives', 'swelling', 'edema', 'redness',
            'inflammation', 'bleeding', 'bruising', 'bruise', 'palpitations',
            'rapid heartbeat', 'weight loss', 'weight gain', 'loss of appetite',
            'increased appetite', 'insomnia', 'difficulty sleeping'
        }

        self.medications = {
            # Pain relievers
            'aspirin', 'ibuprofen', 'advil', 'motrin', 'acetaminophen', 'tylenol',
            'naproxen', 'aleve', 'paracetamol',

            # Antibiotics
            'antibiotic', 'antibiotics', 'penicillin', 'amoxicillin', 'augmentin',
            'azithromycin', 'zithromax', 'doxycycline', 'cephalexin', 'keflex',
            'ciprofloxacin', 'cipro',

            # Respiratory
            'inhaler', 'albuterol', 'ventolin', 'proair', 'steroid', 'prednisone',
            'fluticasone', 'budesonide', 'decongestant', 'antihistamine',

            # Chronic conditions
            'insulin', 'metformin', 'glucophage', 'lisinopril', 'atorvastatin', 'lipitor',
            'simvastatin', 'zocor', 'omeprazole', 'prilosec', 'levothyroxine', 'synthroid',

            # Other common medications
            'multivitamin', 'vitamin', 'vitamins', 'vaccine', 'lozenge', 'cough syrup',
            'benadryl', 'zyrtec', 'claritin', ' Allegra', 'pepto bismol', 'tums'
        }

        self.diagnoses = {
            # Infections
            'covid', 'coronavirus', 'covid-19', 'covid 19',
            'viral infection', 'bacterial infection', 'infection',
            'flu', 'influenza', 'cold', 'common cold',
            'pneumonia', 'bronchitis', 'sinusitis', 'sinus infection',
            'strep throat', 'strep', 'urinary tract infection', 'uti',
            'ear infection', 'eye infection', 'skin infection',

            # Chronic conditions
            'diabetes', 'hypertension', 'high blood pressure', 'heart disease',
            'heart failure', 'coronary artery disease', 'cad',
            'asthma', 'copd', 'chronic obstructive pulmonary disease',
            'arthritis', 'osteoarthritis', 'rheumatoid arthritis',
            'gerd', 'acid reflux', 'gastroesophageal reflux disease',
            'migraine', 'anxiety', 'depression', 'cancer', 'tumor',
            'anemia', 'thyroid disorder', 'hypothyroidism', 'hyperthyroidism'
        }

        self.tests = {
            'blood test', 'blood work', 'lab test', 'cbc', 'complete blood count',
            'chemistry panel', 'metabolic panel', 'urinalysis', 'urine test',
            'x-ray', 'xray', 'chest x-ray', 'mri', 'ct scan', 'cat scan',
            'ultrasound', 'sonogram', 'echocardiogram', 'echo',
            'ecg', 'ekg', 'electrocardiogram', 'stress test', 'holter monitor',
            'pulmonary function test', 'spirometry', 'allergy test',
            'covid test', 'covid swab', 'pcr test', 'rapid test',
            'biopsy', 'endoscopy', 'colonoscopy', 'mammogram', 'pap smear'
        }

        self.body_parts = {
            'head', 'neck', 'shoulder', 'back', 'chest', 'breast',
            'abdomen', 'stomach', 'pelvis', 'hip', 'arm', 'leg',
            'hand', 'foot', 'knee', 'elbow', 'wrist', 'ankle',
            'heart', 'lung', 'liver', 'kidney', 'spleen', 'brain',
            'throat', 'nose', 'ear', 'eye', 'skin', 'bone', 'muscle', 'joint',
            'vocal cords', 'voice box', 'larynx'
        }

        # Common false positives to exclude
        self.exclude_terms = {
            'any', 'some', 'the', 'a', 'an', 'this', 'that', 'these', 'those',
            'my', 'your', 'his', 'her', 'our', 'their', 'have', 'has', 'had',
            'do', 'does', 'did', 'are', 'is', 'was', 'were', 'be', 'been',
            'can', 'could', 'will', 'would', 'should', 'may', 'might', 'must',
            'get', 'got', 'getting', 'feel', 'feeling', 'felt', 'like', 'just',
            'very', 'really', 'quite', 'pretty', 'so', 'too', 'also', 'as well',
            'maybe', 'perhaps', 'possibly', 'probably', 'likely', 'unlikely',
            'seem', 'seems', 'appear', 'appears', 'look', 'looks'
        }

# ============================================================================
# RULE-BASED MEDICAL ENTITY EXTRACTOR
# ============================================================================

class RuleBasedMedicalExtractor:
    """Rule-based medical entity extractor using comprehensive dictionaries"""

    def __init__(self):
        self.dictionary = MedicalDictionary()

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract medical entities using dictionary matching"""
        text_lower = text.lower()

        # Initialize results
        entities = {
            'SYMPTOM': [],
            'MEDICATION': [],
            'DIAGNOSIS': [],
            'TEST': [],
            'BODY_PART': []
        }

        # Extract symptoms
        entities['SYMPTOM'] = self._extract_terms(text_lower, self.dictionary.symptoms)

        # Extract medications
        entities['MEDICATION'] = self._extract_terms(text_lower, self.dictionary.medications)

        # Extract diagnoses
        entities['DIAGNOSIS'] = self._extract_terms(text_lower, self.dictionary.diagnoses)

        # Extract tests
        entities['TEST'] = self._extract_terms(text_lower, self.dictionary.tests)

        # Extract body parts
        entities['BODY_PART'] = self._extract_terms(text_lower, self.dictionary.body_parts)

        # Clean and filter entities
        for category in entities:
            entities[category] = self._clean_entities(entities[category])

        return entities

    def _extract_terms(self, text: str, terms_set: set) -> List[str]:
        """Extract terms from text using the provided set"""
        found_terms = []

        for term in sorted(terms_set, key=len, reverse=True):  # Longest first to avoid partial matches
            # Use word boundaries to avoid partial matches
            pattern = r'\b' + re.escape(term) + r'\b'
            if re.search(pattern, text):
                found_terms.append(term)

        return found_terms

    def _clean_entities(self, entities: List[str]) -> List[str]:
        """Clean and filter entities"""
        cleaned = []
        seen = set()

        for entity in entities:
            # Skip excluded terms
            if entity in self.dictionary.exclude_terms:
                continue

            # Skip very short entities (unless they are common medical terms)
            if len(entity) < 4 and entity not in {'pain', 'rash', 'flu', 'uti', 'cad', 'gerd'}:
                continue

            # Capitalize properly
            if entity not in seen:
                cleaned.append(entity.title())
                seen.add(entity)

        return sorted(cleaned)

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    """Configuration class"""
    def __init__(self, data_dir: str):
        self.project_root = Path.cwd()
        self.raw_data_dir = Path(data_dir)
        self.output_dir = self.project_root / "outputs"

        # Create directories
        self.output_dir.mkdir(exist_ok=True)
        (self.output_dir / "summaries").mkdir(exist_ok=True)
        (self.output_dir / "reports").mkdir(exist_ok=True)

# ============================================================================
# CONVERSATION PARSER
# ============================================================================

class ConversationParser:
    def __init__(self, config: Config):
        self.config = config

    def parse_all_files(self, data_dir: Path) -> List[Dict]:
        """Parse all conversation files"""
        files = sorted(list(data_dir.glob('*.txt')))

        conversations = []
        logger.info(f"📁 Parsing {len(files)} conversation files...")

        for filepath in tqdm(files, desc="Parsing files"):
            conversation = self.parse_file(filepath)
            if conversation['num_turns'] > 0:
                conversations.append(conversation)

        logger.info(f"✅ Successfully parsed {len(conversations)} conversations")
        return conversations

    def parse_file(self, filepath: Path) -> Dict:
        """Parse a single conversation file"""
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read().strip()

            if not content:
                return self._create_empty_conversation(filepath)

            lines = [line.strip() for line in content.split('\n') if line.strip()]
            turns = self._parse_lines(lines)

            return {
                'filename': filepath.name,
                'filepath': str(filepath),
                'turns': turns,
                'num_turns': len(turns),
                'full_text': content
            }

        except Exception as e:
            logger.error(f"Error parsing {filepath}: {e}")
            return self._create_empty_conversation(filepath)

    def _parse_lines(self, lines: List[str]) -> List[Dict]:
        """Parse individual lines into conversation turns"""
        turns = []
        current_speaker = None
        current_text = []

        for line in lines:
            speaker, text = self._parse_line(line)

            if speaker:
                if current_speaker and current_text:
                    turns.append({
                        'speaker': current_speaker,
                        'text': ' '.join(current_text)
                    })
                current_speaker = speaker
                current_text = [text] if text else []
            elif current_speaker and text:
                current_text.append(text)

        if current_speaker and current_text:
            turns.append({
                'speaker': current_speaker,
                'text': ' '.join(current_text)
            })

        return turns

    def _parse_line(self, line: str) -> Tuple[Optional[str], Optional[str]]:
        """Parse a single line for speaker and text"""
        line = line.strip()

        if line.startswith('D:'):
            return 'doctor', line[2:].strip()
        elif line.startswith('P:'):
            return 'patient', line[2:].strip()
        elif line:
            return None, line
        else:
            return None, None

    def _create_empty_conversation(self, filepath: Path) -> Dict:
        return {
            'filename': filepath.name,
            'filepath': str(filepath),
            'turns': [],
            'num_turns': 0,
            'full_text': ''
        }

# ============================================================================
# CLEAN RESULTS GENERATOR
# ============================================================================

class CleanResultsGenerator:
    """Generate clean, accurate clinical summaries"""

    @staticmethod
    def generate_clinical_summary(entities: Dict[str, List[str]], filename: str) -> str:
        """Generate a clean, professional clinical summary"""
        summary = []
        summary.append("=" * 70)
        summary.append("CLINICAL ENTITY EXTRACTION SUMMARY")
        summary.append("=" * 70)
        summary.append(f"File: {filename}")
        summary.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        summary.append("")

        # Patient Symptoms
        summary.append("🧬 PATIENT SYMPTOMS:")
        summary.append("-" * 40)
        symptoms = entities['SYMPTOM']
        if symptoms:
            for symptom in symptoms:
                summary.append(f"  • {symptom}")
        else:
            summary.append("  No symptoms identified")

        # Medications
        summary.append("\n💊 MEDICATIONS & TREATMENTS:")
        summary.append("-" * 40)
        medications = entities['MEDICATION']
        if medications:
            for med in medications:
                summary.append(f"  • {med}")
        else:
            summary.append("  No medications identified")

        # Clinical Diagnoses
        summary.append("\n🏥 CLINICAL IMPRESSIONS:")
        summary.append("-" * 40)
        diagnoses = entities['DIAGNOSIS']
        if diagnoses:
            for diag in diagnoses:
                summary.append(f"  • {diag}")
        else:
            summary.append("  No diagnoses identified")

        # Diagnostic Tests
        summary.append("\n🔬 DIAGNOSTIC TESTS:")
        summary.append("-" * 40)
        tests = entities['TEST']
        if tests:
            for test in tests:
                summary.append(f"  • {test}")
        else:
            summary.append("  No tests mentioned")

        # Body Parts
        summary.append("\n📍 BODY PARTS MENTIONED:")
        summary.append("-" * 40)
        body_parts = entities['BODY_PART']
        if body_parts:
            for part in body_parts:
                summary.append(f"  • {part}")
        else:
            summary.append("  No specific body parts mentioned")

        # Statistics
        summary.append("\n📊 EXTRACTION STATISTICS:")
        summary.append("-" * 40)
        total_entities = sum(len(entities[category]) for category in entities)
        summary.append(f"  Total meaningful entities: {total_entities}")
        summary.append(f"  Symptoms: {len(symptoms)}")
        summary.append(f"  Medications: {len(medications)}")
        summary.append(f"  Diagnoses: {len(diagnoses)}")
        summary.append(f"  Tests: {len(tests)}")
        summary.append(f"  Body parts: {len(body_parts)}")

        summary.append("=" * 70)
        return "\n".join(summary)

# ============================================================================
# MAIN PIPELINE - RULE BASED
# ============================================================================

class RuleBasedMedicalPipeline:
    """Rule-based medical entity extraction pipeline"""

    def __init__(self, data_dir: str):
        self.config = Config(data_dir)
        self.extractor = RuleBasedMedicalExtractor()

    def run(self):
        """Execute the pipeline"""
        logger.info("🚀 Starting Rule-Based Medical Entity Extraction")
        start_time = datetime.now()

        try:
            # Step 1: Parse conversations
            conversations = self.step_parse_conversations()

            # Step 2: Extract entities using rule-based approach
            self.step_extract_entities(conversations)

            # Final summary
            self.generate_final_summary(start_time, len(conversations))

        except Exception as e:
            logger.error(f"Pipeline execution failed: {e}")
            raise

    def step_parse_conversations(self) -> List[Dict]:
        """Step 1: Parse conversations"""
        parser = ConversationParser(self.config)
        return parser.parse_all_files(self.config.raw_data_dir)

    def step_extract_entities(self, conversations: List[Dict]):
        """Step 2: Extract entities using rule-based approach"""
        logger.info("🔍 Extracting medical entities using rule-based approach...")

        for conversation in tqdm(conversations, desc="Processing conversations"):
            # Combine all conversation text
            full_text = conversation['full_text']

            # Extract entities using rule-based approach
            entities = self.extractor.extract_entities(full_text)

            # Generate clean summary
            clinical_summary = CleanResultsGenerator.generate_clinical_summary(
                entities, conversation['filename']
            )

            # Save summary
            filename = Path(conversation['filename']).stem
            output_file = self.config.output_dir / "summaries" / f"{filename}_clinical_summary.txt"

            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(clinical_summary)

        logger.info("✅ All clinical summaries generated successfully")

    def generate_final_summary(self, start_time: datetime, num_conversations: int):
        """Generate final summary"""
        end_time = datetime.now()
        duration = end_time - start_time

        summary = f"""
╔{'═' * 68}╗
║           RULE-BASED EXTRACTION COMPLETE              ║
╚{'═' * 68}╝

📊 RESULTS:
  • Conversations processed: {num_conversations}
  • Method: Rule-based dictionary matching
  • Execution time: {duration}

📁 OUTPUTS:
  • Clean Summaries: {self.config.output_dir / 'summaries'}

✅ PIPELINE COMPLETED SUCCESSFULLY!
  No fragmented entities - Only meaningful medical terms extracted
"""
        logger.info(summary)

# ============================================================================
# EXECUTION
# ============================================================================

def main():
    """Main execution"""
    parser = argparse.ArgumentParser(description='Rule-Based Medical Entity Extraction')
    parser.add_argument('data_dir', type=str,
                       help='Path to directory containing clinical conversation .txt files')

    args = parser.parse_args([DOWNLOAD_PATH]) # Pass DOWNLOAD_PATH as an argument

    if not Path(args.data_dir).exists():
        print(f"❌ Error: Data directory '{args.data_dir}' does not exist")
        sys.exit(1)

    try:
        pipeline = RuleBasedMedicalPipeline(args.data_dir)
        pipeline.run()
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

Processing conversations: 100%|██████████| 270/270 [00:08<00:00, 32.09it/s]
